In [ ]:
# for UI Framework
import streamlit as st
# for handling file operations and delays
import os
import shutil
import time # Import time for a small delay
# for embedding model and vector store
from langchain_community.vectorstores import Chroma
# for loading documents from directories
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
# for splitting text into manageable chunks Next session
from langchain.text_splitter import RecursiveCharacterTextSplitter
# for embedding model
from langchain_community.embeddings import SentenceTransformerEmbeddings

# --- Configuration ---
DATA_DIRECTORY = "./data_source" # Directory where your local TXT/PDF files are stored
PERSIST_DIRECTORY = "./today"  # Directory to store Chroma DB data persistently
COLLECTION_NAME = "testing1"

def ingest_documents():
    """
    Ingests documents from DATA_DIRECTORY, processes them, and stores them in Chroma DB.
    """
    print("Starting document ingestion process...")
    st.info("Initializing embedding model...")

    # Initialize Embedding Model
    try:
        embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
        print("SentenceTransformer Embeddings initialized successfully.")
    except Exception as e:
        print(f"Error initializing SentenceTransformer Embeddings: {e}")
        st.error(f"Error initializing embedding model: {e}")
        return False

    st.info(f"Loading documents from: {DATA_DIRECTORY}")
    documents = []
    try:
        # Load PDF files
        pdf_loader = DirectoryLoader(
            DATA_DIRECTORY,
            glob="**/*.pdf",
            loader_cls=PyPDFLoader,
            silent_errors=True
        )
        pdf_documents = pdf_loader.load()
        documents.extend(pdf_documents)

        # Load TXT files
        txt_loader = DirectoryLoader(
            DATA_DIRECTORY,
            glob="**/*.txt",
            loader_cls=TextLoader,
            silent_errors=True
        )
        txt_documents = txt_loader.load()
        documents.extend(txt_documents)

        print(f"Loaded {len(documents)} documents (PDFs and TXTs).")
        st.success(f"Loaded {len(documents)} documents.")

        if not documents:
            print(f"No documents found in {DATA_DIRECTORY}. Please place your files there.")
            st.warning(f"No documents found in {DATA_DIRECTORY}. Please place your files there.")
            return False

    except Exception as e:
        print(f"Error loading documents: {e}")
        st.error(f"Error loading documents: {e}")
        return False

    # Split documents into chunks
    st.info("Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunked_documents = text_splitter.split_documents(documents)
    print(f"Split documents into {len(chunked_documents)} chunks.")
    st.success(f"Split documents into {len(chunked_documents)} chunks.")

    # Clean existing Chroma DB for a fresh index
    st.info(f"Clearing existing Chroma DB at: {PERSIST_DIRECTORY}...")
    if os.path.exists(PERSIST_DIRECTORY):
        try:
            shutil.rmtree(PERSIST_DIRECTORY)
            print(f"Successfully cleared existing Chroma DB at: {PERSIST_DIRECTORY}")
            time.sleep(0.5) # Small delay to ensure directory is released
        except OSError as e:
            print(f"Error removing existing Chroma DB directory {PERSIST_DIRECTORY}: {e}")
            st.error(f"Error clearing old database: {e}. Please manually delete the '{PERSIST_DIRECTORY}' folder if the issue persists.")
            return False

    # Ensure directory exists, creating if necessary
    try:
        os.makedirs(PERSIST_DIRECTORY, exist_ok=True)
        print(f"Ensured {PERSIST_DIRECTORY} directory exists.")
    except OSError as e:
        print(f"Error creating directory {PERSIST_DIRECTORY}: {e}")
        st.error(f"Error creating database directory: {e}. Check permissions.")
        return False

    # Store in Chroma DB
    st.info(f"Generating embeddings and storing in Chroma DB at {PERSIST_DIRECTORY}...")
    try:
        vectorstore = Chroma.from_documents(
            documents=chunked_documents, # Use chunked documents
            embedding=embeddings,
            persist_directory=PERSIST_DIRECTORY,
            collection_name=COLLECTION_NAME
        )
        vectorstore.persist() # Ensure data is saved to disk
        print("Chroma DB built and persisted successfully!")
        st.success("Chroma DB built and persisted successfully!")
        return True
    except Exception as e:
        print(f"Error building and persisting Chroma DB: {e}")
        st.error(f"Error building and persisting Chroma DB: {e}. Check your files and permissions.")
        return False